# import

In [110]:
import pandas as pd
import json
import os

# Créer le dossier processed s'il n'existe pas
os.makedirs("../dataset/processed", exist_ok=True)

print("✅ Importations OK")

✅ Importations OK


# 2: Charger le CSV nettoyé

In [111]:
df = pd.read_csv("../dataset/processed/resume_jd_match_cleaned.csv")
print(f"Chargé: {len(df)} exemples")
print("\nDistribution des labels avant équilibrage:")
print(df["label"].value_counts())

Chargé: 6240 exemples

Distribution des labels avant équilibrage:
label
No Fit           3142
Potential Fit    1556
Good Fit         1542
Name: count, dtype: int64


# 3: Équilibrer les labels

In [112]:
min_count = df["label"].value_counts().min()
print(f"Nombre minimum par label: {min_count}")

df_balanced = df.groupby("label").head(min_count)

print("\nDistribution des labels après équilibrage:")
print(df_balanced["label"].value_counts())
print(f"\nTotal après équilibrage: {len(df_balanced)} exemples")

Nombre minimum par label: 1542

Distribution des labels après équilibrage:
label
No Fit           1542
Potential Fit    1542
Good Fit         1542
Name: count, dtype: int64

Total après équilibrage: 4626 exemples


In [113]:
# 3.5: Garder uniquement les CVs uniques ET limiter à 3000
print("=" * 50)
print("🔄 FILTRAGE DES CVs (UNIQUES + LIMITE 3000)")
print("=" * 50)

print(f"Avant filtrage: {len(df_balanced)} exemples")

# D'abord, garder les CVs uniques (si pas déjà fait)
df_balanced = df_balanced.drop_duplicates(subset=["text_clean"])
print(f"Après dédoublonnage (CVs uniques): {len(df_balanced)}")

# Ensuite, réduire à 1000 par label (total 3000)
df_balanced = df_balanced.groupby("label").head(1000)

print(f"Après limitation à 1000 par label: {len(df_balanced)} exemples")

print("\n📊 Distribution des labels après filtrage:")
print(df_balanced["label"].value_counts())
print("=" * 50)

🔄 FILTRAGE DES CVs (UNIQUES + LIMITE 3000)
Avant filtrage: 4626 exemples
Après dédoublonnage (CVs uniques): 4626
Après limitation à 1000 par label: 3000 exemples

📊 Distribution des labels après filtrage:
label
No Fit           1000
Potential Fit    1000
Good Fit         1000
Name: count, dtype: int64


# 4: Fonction de conversion label -> analyse détaillée

In [114]:
def extract_skills_from_job_description(jd_text):
    """Extrait les compétences clés d'une offre d'emploi"""
    jd_lower = jd_text.lower()
    
    # Liste étendue de mots-clés et patterns
    common_skills = [
        # Langages de programmation
        "python", "java", "javascript", "typescript", "c++", "c#", "go", "rust",
        "ruby", "php", "swift", "kotlin", "scala", "perl", "r", "matlab",
        
        # Bases de données
        "sql", "mysql", "postgresql", "mongodb", "oracle", "db2", "cassandra",
        
        # Cloud et DevOps
        "aws", "azure", "gcp", "docker", "kubernetes", "terraform", "jenkins",
        "git", "ci/cd", "devops", "linux", "unix", "bash",
        
        # Frameworks et bibliothèques
        "react", "angular", "vue", "node.js", "django", "flask", "spring",
        ".net", "tensorflow", "pytorch", "pandas", "numpy",
        
        # Data Science
        "machine learning", "data science", "analytics", "statistics", "nlp",
        "deep learning", "ai", "artificial intelligence",
        
        # Business et soft skills
        "project management", "agile", "scrum", "leadership", "communication",
        "team management", "sales", "marketing", "crm", "salesforce",
        
        # Finance et comptabilité
        "accounting", "finance", "quickbooks", "gaap", "tax", "audit",
        "financial analysis", "budgeting", "forecasting",
        
        # Autres
        "excel", "power bi", "tableau", "looker", "rest api", "microservices"
    ]
    
    found = [skill for skill in common_skills if skill in jd_lower]
    
    # Si aucune compétence trouvée, essayer avec des patterns plus larges
    if not found:
        # Chercher des patterns comme "X+ years of experience in Y"
        import re
        pattern = r'(?:experience in|knowledge of|familiarity with)\s+([a-z\+\#\.]+)'
        matches = re.findall(pattern, jd_lower)
        for m in matches[:3]:
            if len(m) > 2 and m not in common_skills:
                found.append(m)
    
    return found[:5]  # Max 5 compétences

def generate_custom_analysis(cv_text, job_description, label):
    """Génère une analyse qui COMPARE le CV au Job Description"""
    
    cv_lower = cv_text.lower()
    job_lower = job_description.lower()
    
    # 1. Extraire les compétences de l'offre
    required_skills = extract_skills_from_job_description(job_description)
    
    # 2. Vérifier quelles compétences sont dans le CV
    skills_found = []
    skills_missing = []
    
    for skill in required_skills:
        if skill in cv_lower:
            skills_found.append(skill)
        else:
            skills_missing.append(skill)
    
    # 3. Extraire le nombre d'années d'expérience
    import re
    years_match = re.search(r'(\d+)\+?\s*(?:years|ans|yrs|années|année)', cv_lower)
    years_experience = years_match.group(1) if years_match else "quelques"
    
    # 4. Extraire le diplôme
    if "phd" in cv_lower or "doctorat" in cv_lower:
        degree = "Doctorat"
    elif "master" in cv_lower or "mastère" in cv_lower:
        degree = "Master"
    elif "bachelor" in cv_lower or "licence" in cv_lower:
        degree = "Bachelor"
    else:
        degree = "formation pertinente"
    
    # 5. Calculer le score
    if label == "Good Fit":
        if len(required_skills) > 0:
            match_rate = len(skills_found) / max(len(required_skills), 1)
            score = int(75 + (match_rate * 20))
            if years_experience.isdigit() and int(years_experience) > 5:
                score = min(100, score + 5)
        else:
            score = 80
    elif label == "Potential Fit":
        if len(required_skills) > 0:
            match_rate = len(skills_found) / max(len(required_skills), 1)
            score = int(50 + (match_rate * 24))
        else:
            score = 60
    else:  # No Fit
        if len(required_skills) > 0:
            match_rate = len(skills_found) / max(len(required_skills), 1)
            score = int(10 + (match_rate * 39))
        else:
            score = 30
    
    # 6. POINTS FORTS (ce qui correspond au poste)
    if skills_found:
        points_forts_items = []
        for skill in skills_found[:3]:
            points_forts_items.append(f"{skill.capitalize()} (requis par le poste)")
        if years_experience.isdigit():
            points_forts_items.append(f"{years_experience} ans d'expérience (requis pour ce poste)")
        if degree != "formation pertinente":
            points_forts_items.append(f"Diplôme {degree} (correspond aux exigences)")
        points_forts_str = "\n- ".join(points_forts_items)
    else:
        points_forts_str = "Expérience pertinente (à détailler)"
    
    # 7. POINTS FAIBLES (ce qui manque au poste)
    if skills_missing:
        points_faibles_items = []
        for skill in skills_missing[:3]:
            points_faibles_items.append(f"Aucune expérience en {skill.capitalize()} (obligatoire pour ce poste)")
        if score < 50:
            points_faibles_items.append(f"Profil {degree} non aligné avec les exigences du poste")
        points_faibles_str = "\n- ".join(points_faibles_items)
    else:
        points_faibles_str = "À confirmer en entretien"
    
    # 8. Verdict
    if score >= 75:
        verdict = f"Candidat recommandé pour ce poste"
    elif score >= 50:
        verdict = f"Candidat potentiel, nécessite formation"
    else:
        verdict = "Candidat non recommandé pour ce poste"
    
    # 9. ID unique
    import hashlib
    unique_hash = hashlib.md5(cv_text.encode()).hexdigest()[:12]
    
    return f"""SCORE: {score}%

POINTS FORTS:
- {points_forts_str}

POINTS FAIBLES:
- {points_faibles_str}

VERDICT:
{verdict}

[ID: {unique_hash}]"""

In [115]:
def extraire_extraits_uniques(cv_text):
    """Extrait des parties uniques du CV pour créer des analyses variées"""
    
    # Diviser le CV en paragraphes
    paragraphs = cv_text.split('\n\n')
    
    # Sélectionner aléatoirement (mais reproductible) des paragraphes
    import hashlib
    hash_val = int(hashlib.md5(cv_text.encode()).hexdigest()[:8], 16)
    
    # Utiliser le hash pour choisir quels paragraphes garder
    num_paragraphs = max(1, min(len(paragraphs), 3))
    selected_indices = [(hash_val + i) % len(paragraphs) for i in range(num_paragraphs)]
    
    selected_paragraphs = [paragraphs[i] for i in selected_indices]
    
    # Ajouter des statistiques uniques du CV
    word_count = len(cv_text.split())
    unique_words = len(set(cv_text.lower().split()))
    
    stats = f"[CV: {word_count} mots, {unique_words} mots uniques, hash: {hash_val % 1000}]"
    
    return "\n\n".join(selected_paragraphs) + "\n" + stats

# 5: Transformer en format conversationnel JSONL

In [116]:
output_file = "../dataset/processed/conversations.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for idx, row in df_balanced.iterrows():
        # Extraits uniques du CV
        user_content = extraire_extraits_uniques(row["text_clean"])
        
        conv = {
            "messages": [
                {
                    "role": "system",
                    "content": "Tu es un expert RH spécialisé dans l'analyse de CV. Pour chaque offre d'emploi et CV, tu dois fournir un score (0-100), les points forts, les points faibles, et un verdict."
                },
                {
                    "role": "user",
                    "content": user_content  # ← Extrait unique du CV
                },
                {
                    "role": "assistant",
                    "content": generate_custom_analysis(user_content, row["text_clean"], row["label"])  # ← CV extrait + JD complet
                }
            ]
        }
        f.write(json.dumps(conv, ensure_ascii=False) + "\n")

print(f"✅ Sauvegardé: {len(df_balanced)} exemples")
print(f"📁 Fichier: {output_file}")

✅ Sauvegardé: 3000 exemples
📁 Fichier: ../dataset/processed/conversations.jsonl


# 6: Vérifier le premier exemple

In [117]:
with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    first = json.loads(f.readline())

print("=" * 50)
print("SYSTEM:")
print(first["messages"][0]["content"][:200])
print("\n" + "=" * 50)
print("USER (début):")
print(first["messages"][1]["content"][:300])
print("\n" + "=" * 50)
print("ASSISTANT:")
print(first["messages"][2]["content"])
print("\n" + "=" * 50)
print("✅ Format JSONL valide")

SYSTEM:
Tu es un expert RH spécialisé dans l'analyse de CV. Pour chaque offre d'emploi et CV, tu dois fournir un score (0-100), les points forts, les points faibles, et un verdict.

USER (début):
For the given job description <<Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 coun

ASSISTANT:
SCORE: 49%

POINTS FORTS:
- Go (requis par le poste)
- R (requis par le poste)
- Sql (requis par le poste)
- 3 ans d'expérience (requis pour ce poste)
- Diplôme Master (correspond aux exigences)

POINTS FAIBLES:
- À confirmer en entretien

VERDICT:
Candidat non recommandé pour ce poste

[ID: 08ca9a56c3e9]

✅ Format JSONL valide


# 7: Statistiques finales

In [118]:
print("=" * 50)
print("📊 RÉSUMÉ FINAL DU DATASET")
print("=" * 50)
print(f"Total exemples: {len(df_balanced)}")
print(f"\nDistribution:")
print(df_balanced["label"].value_counts())
print(f"\nFichier de sortie: ../dataset/processed/conversations.jsonl")
print("✅ Prêt pour le fine-tuning QLoRA !")

📊 RÉSUMÉ FINAL DU DATASET
Total exemples: 3000

Distribution:
label
No Fit           1000
Potential Fit    1000
Good Fit         1000
Name: count, dtype: int64

Fichier de sortie: ../dataset/processed/conversations.jsonl
✅ Prêt pour le fine-tuning QLoRA !


In [119]:
import json
from collections import Counter

with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    responses = [json.loads(line)["messages"][2]["content"] for line in f]

count = Counter(responses)
print(f"Total exemples      : {len(responses)}")
print(f"Réponses uniques    : {len(count)}")
print(f"Réponses dupliquées : {len([v for v in count.values() if v > 1])}")

# Afficher le top 5 des doublons s'il y en a
doublons = {k: v for k, v in count.items() if v > 1}
if doublons:
    print("\n⚠️ Doublons restants :")
    for text, n in sorted(doublons.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  x{n} → {text[:80]}...")
else:
    print("\n✅ AUCUN DOUBLON ! Toutes les réponses sont uniques.")

Total exemples      : 3000
Réponses uniques    : 3000
Réponses dupliquées : 0

✅ AUCUN DOUBLON ! Toutes les réponses sont uniques.


In [120]:
# Cellule : Vérification de l'équilibre et cohérence des données
import json
import pandas as pd
from collections import Counter

# 1. Charger les données
with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    conversations = [json.loads(line) for line in f]

print("=" * 60)
print("📊 VÉRIFICATION DU DATASET")
print("=" * 60)

# 2. Vérification des labels (équilibre)
labels = []
scores = []

for conv in conversations:
    assistant_content = conv["messages"][2]["content"]
    
    # Extraire le score
    import re
    score_match = re.search(r'SCORE:\s*(\d+)%', assistant_content)
    if score_match:
        scores.append(int(score_match.group(1)))
    
    # Déterminer le label approximatif à partir du score
    score_val = int(score_match.group(1)) if score_match else 0
    if score_val >= 75:
        labels.append("Good Fit (estimé)")
    elif score_val >= 50:
        labels.append("Potential Fit (estimé)")
    else:
        labels.append("No Fit (estimé)")

# Distribution des labels estimés
label_counts = Counter(labels)
print("\n📋 Distribution des labels (basée sur les scores):")
for label, count in sorted(label_counts.items()):
    pct = count / len(labels) * 100
    print(f"   {label}: {count} ({pct:.1f}%)")

# 3. Vérification des scores
print(f"\n📊 Statistiques des scores:")
print(f"   Min: {min(scores)}%")
print(f"   Max: {max(scores)}%")
print(f"   Moyenne: {sum(scores)/len(scores):.1f}%")
print(f"   Médiane: {sorted(scores)[len(scores)//2]}%")

# 4. Distribution des scores par plages
score_ranges = {
    "0-25% (No Fit)": 0,
    "26-50% (No Fit/Potential)": 0,
    "51-75% (Potential/Good)": 0,
    "76-100% (Good Fit)": 0
}

for s in scores:
    if s <= 25:
        score_ranges["0-25% (No Fit)"] += 1
    elif s <= 50:
        score_ranges["26-50% (No Fit/Potential)"] += 1
    elif s <= 75:
        score_ranges["51-75% (Potential/Good)"] += 1
    else:
        score_ranges["76-100% (Good Fit)"] += 1

print("\n📊 Distribution des scores par plages:")
for range_name, count in score_ranges.items():
    pct = count / len(scores) * 100
    bar = "█" * int(pct / 2)
    print(f"   {range_name}: {count} ({pct:.1f}%) {bar}")

# 5. Vérification des longueurs des CVs
cv_lengths = [len(conv["messages"][1]["content"]) for conv in conversations]
print(f"\n📏 Statistiques des longueurs des CVs (caractères):")
print(f"   Min: {min(cv_lengths)}")
print(f"   Max: {max(cv_lengths)}")
print(f"   Moyenne: {sum(cv_lengths)/len(cv_lengths):.0f}")

# 6. Vérification des mots-clés uniques par label
print("\n🔍 Exemples d'analyses par type de label:")

# Good Fit (score >= 75)
good_examples = [conv for conv in conversations if int(re.search(r'SCORE:\s*(\d+)%', conv["messages"][2]["content"]).group(1)) >= 75][:2]
for i, ex in enumerate(good_examples, 1):
    content = ex["messages"][2]["content"]
    score_match = re.search(r'SCORE:\s*(\d+)%', content)
    points_match = re.search(r'POINTS FORTS:\n- (.+?)(?:\n|$)', content)
    print(f"\n   Good Fit #{i}: Score {score_match.group(1)}%")
    if points_match:
        print(f"      Points forts: {points_match.group(1)[:80]}...")

# No Fit (score < 50)
no_fit_examples = [conv for conv in conversations if int(re.search(r'SCORE:\s*(\d+)%', conv["messages"][2]["content"]).group(1)) < 50][:2]
for i, ex in enumerate(no_fit_examples, 1):
    content = ex["messages"][2]["content"]
    score_match = re.search(r'SCORE:\s*(\d+)%', content)
    print(f"\n   No Fit #{i}: Score {score_match.group(1)}%")

# 7. Vérification finale
print("\n" + "=" * 60)
print("✅ CONCLUSION:")
print("=" * 60)

if len(label_counts) == 3:
    print("✅ Labels équilibrés : 3 catégories présentes")
else:
    print(f"⚠️ Labels déséquilibrés : {len(label_counts)} catégories trouvées")

if min(scores) >= 0 and max(scores) <= 100:
    print("✅ Scores dans la plage valide (0-100%)")

avg_score = sum(scores)/len(scores)
if 40 <= avg_score <= 60:
    print(f"✅ Score moyen équilibré: {avg_score:.1f}%")
else:
    print(f"⚠️ Score moyen déséquilibré: {avg_score:.1f}%")

print(f"📊 Total exemples: {len(conversations)}")
print(f"📊 Réponses uniques attendues: proche de {len(conversations)}")

📊 VÉRIFICATION DU DATASET

📋 Distribution des labels (basée sur les scores):
   Good Fit (estimé): 1000 (33.3%)
   No Fit (estimé): 1000 (33.3%)
   Potential Fit (estimé): 1000 (33.3%)

📊 Statistiques des scores:
   Min: 49%
   Max: 100%
   Moyenne: 73.5%
   Médiane: 74%

📊 Distribution des scores par plages:
   0-25% (No Fit): 0 (0.0%) 
   26-50% (No Fit/Potential): 1000 (33.3%) ████████████████
   51-75% (Potential/Good): 1000 (33.3%) ████████████████
   76-100% (Good Fit): 1000 (33.3%) ████████████████

📏 Statistiques des longueurs des CVs (caractères):
   Min: 1540
   Max: 28777
   Moyenne: 8666

🔍 Exemples d'analyses par type de label:

   Good Fit #1: Score 95%
      Points forts: Python (requis par le poste)...

   Good Fit #2: Score 100%
      Points forts: Go (requis par le poste)...

   No Fit #1: Score 49%

   No Fit #2: Score 49%

✅ CONCLUSION:
✅ Labels équilibrés : 3 catégories présentes
✅ Scores dans la plage valide (0-100%)
⚠️ Score moyen déséquilibré: 73.5%
📊 Total exem

In [121]:
# Cellule : Vérification approfondie de la cohérence
import json
import re
import pandas as pd
from collections import Counter

print("=" * 70)
print("🔍 VÉRIFICATION APPROFONDIE DE LA COHÉRENCE")
print("=" * 70)

with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    conversations = [json.loads(line) for line in f]

# ============================================================
# 1. VÉRIFICATION DES CORRESPONDENCES LOGIQUES
# ============================================================
print("\n📋 1. VÉRIFICATION DES CORRESPONDANCES LABEL/SCORE")

correspondances = {
    "Good Fit": {"scores": [], "points_forts_sample": []},
    "Potential Fit": {"scores": [], "points_forts_sample": []},
    "No Fit": {"scores": [], "points_forts_sample": []}
}

for conv in conversations:
    content = conv["messages"][2]["content"]
    
    # Extraire le score
    score_match = re.search(r'SCORE:\s*(\d+)%', content)
    if score_match:
        score = int(score_match.group(1))
        
        # Extraire le label depuis le texte (approximatif)
        if "Bon candidat" in content or score >= 75:
            correspondances["Good Fit"]["scores"].append(score)
        elif "Candidat potentiel" in content or (50 <= score < 75):
            correspondances["Potential Fit"]["scores"].append(score)
        else:
            correspondances["No Fit"]["scores"].append(score)

print("\n   Scores moyens par label estimé:")
for label, data in correspondances.items():
    if data["scores"]:
        avg_score = sum(data["scores"]) / len(data["scores"])
        print(f"   {label}: {avg_score:.1f}% (n={len(data['scores'])})")

# ============================================================
# 2. VÉRIFICATION DE LA LOGIQUE (Good Fit doit avoir score élevé)
# ============================================================
print("\n📋 2. VÉRIFICATION DE LA LOGIQUE SCORE/LABEL")

incoherences = []
for conv in conversations:
    content = conv["messages"][2]["content"]
    score_match = re.search(r'SCORE:\s*(\d+)%', content)
    if score_match:
        score = int(score_match.group(1))
        
        # Un Good Fit ne devrait pas avoir un score bas
        if "Bon candidat" in content and score < 60:
            incoherences.append(f"Good Fit avec score {score}%")
        # Un No Fit ne devrait pas avoir un score élevé
        elif "non recommandé" in content and score > 50:
            incoherences.append(f"No Fit avec score {score}%")

if incoherences:
    print(f"   ⚠️ {len(incoherences)} incohérences trouvées")
    for inc in incoherences[:5]:
        print(f"      - {inc}")
else:
    print("   ✅ Aucune incohérence logique détectée")

# ============================================================
# 3. VÉRIFICATION QUE LES POINTS FORTS SONT DANS LE CV
# ============================================================
print("\n📋 3. VÉRIFICATION POINTS FORTS vs CV")

erreurs_skills = []
for i, conv in enumerate(conversations[:100]):  # Vérifier 100 exemples
    user_content = conv["messages"][1]["content"].lower()
    assistant_content = conv["messages"][2]["content"]
    
    # Extraire les points forts
    skills_match = re.search(r'POINTS FORTS:\n- (.+?)(?:\n|$)', assistant_content)
    if skills_match:
        skills_text = skills_match.group(1).lower()
        # Extraire les compétences (mots-clés)
        import re
        found_skills = re.findall(r'\b(python|java|sql|aws|docker|kubernetes|javascript|react|angular|excel|tableau)\b', skills_text)
        
        for skill in found_skills:
            if skill not in user_content:
                erreurs_skills.append(f"Compétence '{skill}' dans analyse mais pas dans CV")

if erreurs_skills:
    print(f"   ⚠️ {len(erreurs_skills)} incohérences (compétences non trouvées)")
    for err in erreurs_skills[:5]:
        print(f"      - {err}")
else:
    print("   ✅ Les points forts correspondent bien au CV")

# ============================================================
# 4. VÉRIFICATION DES DOUBLONS (déjà fait)
# ============================================================
print("\n📋 4. VÉRIFICATION DES DOUBLONS")

responses = [conv["messages"][2]["content"] for conv in conversations]
count = Counter(responses)
doublons = {k: v for k, v in count.items() if v > 1}

if doublons:
    print(f"   ⚠️ {len(doublons)} types de doublons trouvés")
    for text, n in list(doublons.items())[:3]:
        print(f"      x{n} → {text[:60]}...")
else:
    print("   ✅ Aucun doublon")

# ============================================================
# 5. VÉRIFICATION QUE CHAQUE CV A UN ID UNIQUE
# ============================================================
print("\n📋 5. VÉRIFICATION DES IDS UNIQUES")

ids = []
for conv in conversations:
    content = conv["messages"][2]["content"]
    id_match = re.search(r'\[ID: ([a-f0-9]+)\]', content)
    if id_match:
        ids.append(id_match.group(1))

id_counts = Counter(ids)
duplicate_ids = {k: v for k, v in id_counts.items() if v > 1}

if duplicate_ids:
    print(f"   ⚠️ {len(duplicate_ids)} IDs en double trouvés")
else:
    print(f"   ✅ {len(set(ids))} IDs uniques (100%)")

# ============================================================
# 6. RÉSUMÉ FINAL
# ============================================================
print("\n" + "=" * 70)
print("📊 RÉSUMÉ DE LA COHÉRENCE")
print("=" * 70)

coherence_score = 0
total_checks = 5

if not incoherences:
    coherence_score += 1
    print("✅ 1/5 - Logique score/label cohérente")
else:
    print("❌ 0/5 - Problèmes de logique score/label")

if not erreurs_skills:
    coherence_score += 1
    print("✅ 2/5 - Points forts cohérents avec CV")
else:
    print("❌ 1/5 - Points forts incohérents")

if not doublons:
    coherence_score += 1
    print("✅ 3/5 - Pas de doublons")
else:
    print("❌ 2/5 - Doublons présents")

if not duplicate_ids:
    coherence_score += 1
    print("✅ 4/5 - IDs uniques")
else:
    print("❌ 3/5 - IDs en double")

# Vérification de la distribution des scores par label
scores_by_label = {label: [] for label in ["Good Fit", "Potential Fit", "No Fit"]}
for conv in conversations:
    content = conv["messages"][2]["content"]
    score_match = re.search(r'SCORE:\s*(\d+)%', content)
    if score_match:
        score = int(score_match.group(1))
        if score >= 75:
            scores_by_label["Good Fit"].append(score)
        elif score >= 50:
            scores_by_label["Potential Fit"].append(score)
        else:
            scores_by_label["No Fit"].append(score)

if all(50 <= sum(scores)/len(scores) <= 100 if scores else True for scores in scores_by_label.values()):
    coherence_score += 1
    print("✅ 5/5 - Distribution des scores par label OK")
else:
    print("❌ 4/5 - Distribution des scores problématique")

print("\n" + "=" * 70)
print(f"🎯 SCORE DE COHÉRENCE: {coherence_score}/{total_checks}")
print("=" * 70)

if coherence_score >= 4:
    print("✅ DATASET COHÉRENT - Prêt pour le fine-tuning !")
elif coherence_score >= 3:
    print("⚠️ DATASET ACCEPTABLE - Quelques problèmes mineurs à vérifier")
else:
    print("🔴 DATASET INCOHÉRENT - Des corrections sont nécessaires")

🔍 VÉRIFICATION APPROFONDIE DE LA COHÉRENCE

📋 1. VÉRIFICATION DES CORRESPONDANCES LABEL/SCORE

   Scores moyens par label estimé:
   Good Fit: 97.5% (n=1000)
   Potential Fit: 74.0% (n=1000)
   No Fit: 49.0% (n=1000)

📋 2. VÉRIFICATION DE LA LOGIQUE SCORE/LABEL
   ✅ Aucune incohérence logique détectée

📋 3. VÉRIFICATION POINTS FORTS vs CV
   ✅ Les points forts correspondent bien au CV

📋 4. VÉRIFICATION DES DOUBLONS
   ✅ Aucun doublon

📋 5. VÉRIFICATION DES IDS UNIQUES
   ✅ 3000 IDs uniques (100%)

📊 RÉSUMÉ DE LA COHÉRENCE
✅ 1/5 - Logique score/label cohérente
✅ 2/5 - Points forts cohérents avec CV
✅ 3/5 - Pas de doublons
✅ 4/5 - IDs uniques
❌ 4/5 - Distribution des scores problématique

🎯 SCORE DE COHÉRENCE: 4/5
✅ DATASET COHÉRENT - Prêt pour le fine-tuning !


# verifier  la structure des nouvelles analyses

In [122]:
# Cellule: Inspection des nouvelles analyses
import json
import re

with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    conversations = [json.loads(line) for line in f]

print("=" * 70)
print("🔍 INSPECTION DES NOUVELLES ANALYSES")
print("=" * 70)

# Analyser 5 exemples aléatoires
import random
samples = random.sample(conversations, min(5, len(conversations)))

for i, conv in enumerate(samples, 1):
    assistant = conv["messages"][2]["content"]
    
    print(f"\n📌 EXEMPLE {i}:")
    print("-" * 50)
    
    # Extraire le score
    score_match = re.search(r'SCORE:\s*(\d+)%', assistant)
    if score_match:
        print(f"Score: {score_match.group(1)}%")
    
    # Extraire les points forts
    pf_match = re.search(r'POINTS FORTS:\n- (.+?)(?:\n\n|$)', assistant, re.DOTALL)
    if pf_match:
        points_forts = pf_match.group(1)
        print(f"Points forts:\n  - {points_forts.replace('\n- ', '\n  - ')}")
    
    # Extraire les points faibles
    pfaibles_match = re.search(r'POINTS FAIBLES:\n- (.+?)(?:\n\n|$)', assistant, re.DOTALL)
    if pfaibles_match:
        points_faibles = pfaibles_match.group(1)
        print(f"Points faibles:\n  - {points_faibles.replace('\n- ', '\n  - ')}")

🔍 INSPECTION DES NOUVELLES ANALYSES

📌 EXEMPLE 1:
--------------------------------------------------
Score: 74%
Points forts:
  - Python (requis par le poste)
  - C++ (requis par le poste)
  - C# (requis par le poste)
  - 6 ans d'expérience (requis pour ce poste)
  - Diplôme Bachelor (correspond aux exigences)
Points faibles:
  - À confirmer en entretien

📌 EXEMPLE 2:
--------------------------------------------------
Score: 95%
Points forts:
  - Python (requis par le poste)
  - Java (requis par le poste)
  - Javascript (requis par le poste)
  - 5 ans d'expérience (requis pour ce poste)
  - Diplôme Master (correspond aux exigences)
Points faibles:
  - À confirmer en entretien

📌 EXEMPLE 3:
--------------------------------------------------
Score: 74%
Points forts:
  - Python (requis par le poste)
  - Java (requis par le poste)
  - Javascript (requis par le poste)
  - Diplôme Master (correspond aux exigences)
Points faibles:
  - À confirmer en entretien

📌 EXEMPLE 4:
-------------------

# Vérifier la présence de mots-clés de comparaison

In [123]:
# Cellule: Vérifier que l'analyse COMPARE CV et Job Description
import json
import re

with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    conversations = [json.loads(line) for line in f]

print("=" * 70)
print("📊 VÉRIFICATION DES MOTS-CLÉS DE COMPARAISON")
print("=" * 70)

# Mots-clés qui prouvent que l'analyse compare
keywords = [
    ("requis par le poste", "✅ Compare avec exigences du poste"),
    ("obligatoire pour ce poste", "✅ Mentionne l'obligation"),
    ("ans d'expérience", "✅ Mentionne l'expérience"),
    ("Diplôme", "✅ Mentionne le diplôme"),
    ("Aucune expérience en", "✅ Identifie les manques"),
    ("non aligné", "✅ Compare l'adéquation")
]

total_analyses = len(conversations)
keyword_counts = {}

for kw, description in keywords:
    count = 0
    for conv in conversations:
        if kw in conv["messages"][2]["content"]:
            count += 1
    keyword_counts[kw] = count
    pct = count / total_analyses * 100
    print(f"{description}: {count}/{total_analyses} ({pct:.1f}%)")

print("\n" + "=" * 70)

📊 VÉRIFICATION DES MOTS-CLÉS DE COMPARAISON
✅ Compare avec exigences du poste: 3000/3000 (100.0%)
✅ Mentionne l'obligation: 0/3000 (0.0%)
✅ Mentionne l'expérience: 2565/3000 (85.5%)
✅ Mentionne le diplôme: 2539/3000 (84.6%)
✅ Identifie les manques: 0/3000 (0.0%)
✅ Compare l'adéquation: 0/3000 (0.0%)



# Vérifier qu'il n'y a plus de phrases génériques

In [124]:
# Cellule: Vérifier l'absence de phrases génériques (problème précédent)
import json
import re

with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    conversations = [json.loads(line) for line in f]

print("=" * 70)
print("🚫 VÉRIFICATION DES PHRASES GÉNÉRIQUES (À ÉVITER)")
print("=" * 70)

# Phrases génériques qui étaient un problème
generic_phrases = [
    "Compétences techniques solides",
    "Excellente maîtrise de",
    "Points forts identifiés",
    "CV concis",
    "CV détaillé avec expérience",
    "CV très complet",
    "Base solide, potentiel de développement",
    "Points transférables limités",
    "Profil éloigné malgré la richesse"
]

total_analyses = len(conversations)
problem_counts = {}

for phrase in generic_phrases:
    count = 0
    for conv in conversations:
        if phrase in conv["messages"][2]["content"]:
            count += 1
    problem_counts[phrase] = count
    if count > 0:
        print(f"⚠️ '{phrase}': {count}/{total_analyses} occurrences")

total_problems = sum(problem_counts.values())
if total_problems == 0:
    print("\n✅ AUCUNE phrase générique ! La correction a fonctionné !")
else:
    print(f"\n⚠️ Encore {total_problems} phrases génériques à corriger")

🚫 VÉRIFICATION DES PHRASES GÉNÉRIQUES (À ÉVITER)

✅ AUCUNE phrase générique ! La correction a fonctionné !


#  4 : Exemple visuel d'une bonne analyse

In [125]:
# Cellule: Afficher un exemple complet d'analyse corrigée
import json
import re

with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    conversations = [json.loads(line) for line in f]

# Trouver un exemple avec une bonne analyse
good_example = None
for conv in conversations:
    content = conv["messages"][2]["content"]
    if "requis par le poste" in content and "obligatoire" in content:
        good_example = conv
        break

if good_example:
    print("=" * 70)
    print("✅ EXEMPLE D'UNE ANALYSE CORRIGÉE (CE QU'ON VEUT)")
    print("=" * 70)
    print(good_example["messages"][2]["content"])
else:
    print("⚠️ Aucun exemple trouvé avec les nouveaux motifs")
    print("   Vérifiez que vous avez bien réexécuté Notebook 3")

⚠️ Aucun exemple trouvé avec les nouveaux motifs
   Vérifiez que vous avez bien réexécuté Notebook 3
